In [1]:
# 1. Instala dependências necessárias no sistema
!apt-get update -qq --fix-missing
!apt-get install -y clang libclang-dev z3 curl -qq

# 2. Baixa a versão mais recente diretamente do repositório via API do GitHub
!curl -s https://api.github.com/repos/esbmc/esbmc/releases/latest \
  | grep "browser_download_url.*linux" \
  | cut -d : -f 2,3 \
  | tr -d \" \
  | head -n 1 \
  | wget -i - -O esbmc-linux.zip

# 3. Extrai e instala o binário
!unzip -q -o esbmc-linux.zip -d /tmp/esbmc_out || tar -xzf esbmc-linux.zip -C /tmp/esbmc_out
!find /tmp/esbmc_out -name "esbmc" -exec cp {} /usr/local/bin/ \;
!chmod +x /usr/local/bin/esbmc

# 4. Confirma a instalação
!esbmc --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-yaml.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../00-python3-yaml_5.4.1-1ubuntu1_amd64.deb ...
Unpacking python3-yaml (5.4.1-1ubuntu1) ...
Selecting previously unselected package binfmt-support.
Preparing to unpack .../01-binfmt-support_2.2.1-2_amd64.deb ...
Unpacking binfmt-support (2.2.1-2) ...
Selecting previously unselected package libllvm14:amd64.
Preparing to unpack .../02-libllvm14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libllvm14:amd64 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libclang-cpp14.
Preparing to unpack .../03-libclang-cpp14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libclang-cpp14 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libgc

In [3]:
from google.colab import files
uploaded = files.upload() # Escolha o arquivo experimento_safety_esbmc.json

Saving experimento_safety_esbmc.json to experimento_safety_esbmc.json


In [ ]:
import json

# 1. Carrega os traces originais
with open("experimento_safety_esbmc.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

ID_APPROVED = 1
ID_CANCELLED = 2
ID_OUTROS = 0

def converter_trace(trace):
    res = []
    for ev in trace:
        if ev == "A_APPROVED":
            res.append(ID_APPROVED)
        elif ev == "A_CANCELLED":
            res.append(ID_CANCELLED)
        else:
            res.append(ID_OUTROS)
    return res

def gerar_codigo_unrolled(list_of_traces):
    lines = []
    lines.append("def testar_traces():")

    # Gera a verificação em linha para cada trace sem criar loops 'for'
    for t_idx, trace in enumerate(list_of_traces):
        lines.append(f"    # Trace {t_idx}")
        lines.append("    aprovado = 0")
        for ev in trace:
            if ev == ID_APPROVED:
                lines.append("    aprovado = 1")
            elif ev == ID_CANCELLED:
                lines.append("    assert aprovado == 0, 'VIOLACAO DE SAFETY'")

    lines.append("\nif __name__ == '__main__':")
    lines.append("    testar_traces()")

    return "\n".join(lines)

# Processa todas as sequências (ou limite em 100 se quiser teste ultra-rápido)
aalpy_num = [converter_trace(t) for t in dados["aalpy_sequences"]]
agilkia_num = [converter_trace(t) for t in dados["agilkia_sequences"]]

with open("check_aalpy.py", "w") as f:
    f.write(gerar_codigo_unrolled(aalpy_num))

with open("check_agilkia.py", "w") as f:
    f.write(gerar_codigo_unrolled(agilkia_num))

print("Scripts sem loops gerados com sucesso!")

Scripts sem loops gerados com sucesso!


In [4]:
import json

# 1. Carrega as sequências
with open("experimento_safety_esbmc.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

# 2. Funções de conversão e geração
def converter_trace_scanner(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip()
        if 'scanner' in ev:    res.append(1)
        elif 'payer' in ev:    res.append(2)
        elif 'debloquer' in ev: res.append(3)
        elif 'transmission' in ev: res.append(4)
        else:                  res.append(0)
    return res

def gerar_codigo_safety_scanner(list_of_traces):
    lines = ["def testar_traces():"]
    for t_idx, trace in enumerate(list_of_traces):
        lines.append(f"    # Trace {t_idx}")
        lines.append("    houve_scanner = 0")
        for ev in trace:
            if ev == 1:
                lines.append("    houve_scanner = 1")
            elif ev == 2:
                lines.append("    assert houve_scanner == 1, 'SAFETY: payer sem scanner anterior'")
    lines.append("\nif __name__ == '__main__':")
    lines.append("    testar_traces()")
    return "\n".join(lines)

# 3. Converte e gera os arquivos
aalpy_num   = [converter_trace_scanner(t) for t in dados["aalpy_sequences"]]
agilkia_num = [converter_trace_scanner(t) for t in dados["agilkia_sequences"]]

with open("check_aalpy.py", "w") as f:
    f.write(gerar_codigo_safety_scanner(aalpy_num))

with open("check_agilkia.py", "w") as f:
    f.write(gerar_codigo_safety_scanner(agilkia_num))

print("Arquivos gerados!")
print(f"Traces AALpy   : {len(aalpy_num)}")
print(f"Traces Agilkia : {len(agilkia_num)}")

Arquivos gerados!
Traces AALpy   : 5
Traces Agilkia : 4818


In [5]:
!echo "=== AALpy ===" && esbmc check_aalpy.py
!echo "=== Agilkia ===" && esbmc check_agilkia.py

=== AALpy ===
ESBMC version 8.4.0 64-bit x86_64 linux
Target: 64-bit little-endian x86_64-unknown-linux with esbmclibc
[PROGRESS] Parsing check_aalpy.py
  Install with: pip install mypy  or  pipx install mypy
[PROGRESS] Converting
[PROGRESS] Generating GOTO Program
GOTO program creation time: 1.732s
GOTO program processing time: 0.029s
[PROGRESS] Starting Bounded Model Checking
Symex completed in: 0.003s (78 assignments)
Caching time: 0.000s (removed 0 assertions)
Slicing time: 0.000s (removed 78 assignments)
Generated 0 VCC(s), 0 remaining after simplification (0 assignments)
BMC program time: 0.003s

VERIFICATION SUCCESSFUL
=== Agilkia ===
ESBMC version 8.4.0 64-bit x86_64 linux
Target: 64-bit little-endian x86_64-unknown-linux with esbmclibc
[PROGRESS] Parsing check_agilkia.py
  Install with: pip install mypy  or  pipx install mypy
[PROGRESS] Converting
[PROGRESS] Generating GOTO Program
GOTO program creation time: 235.503s
GOTO program processing time: 0.212s
[PROGRESS] Starting Bo

In [ ]:
!esbmc check_aalpy.py

ESBMC version 8.4.0 64-bit x86_64 linux
Target: 64-bit little-endian x86_64-unknown-linux with esbmclibc
[PROGRESS] Parsing check_aalpy.py
  Install with: pip install mypy  or  pipx install mypy
[PROGRESS] Converting
[PROGRESS] Generating GOTO Program
GOTO program creation time: 1.917s
GOTO program processing time: 0.023s
[PROGRESS] Starting Bounded Model Checking
Symex completed in: 0.002s (74 assignments)
Caching time: 0.000s (removed 0 assertions)
Slicing time: 0.000s (removed 74 assignments)
Generated 0 VCC(s), 0 remaining after simplification (0 assignments)
BMC program time: 0.002s

VERIFICATION SUCCESSFUL


In [ ]:
!esbmc check_agilkia.py

ESBMC version 8.4.0 64-bit x86_64 linux
Target: 64-bit little-endian x86_64-unknown-linux with esbmclibc
[PROGRESS] Parsing check_agilkia.py
  Install with: pip install mypy  or  pipx install mypy
[PROGRESS] Converting
[PROGRESS] Generating GOTO Program
GOTO program creation time: 4.246s
GOTO program processing time: 0.053s
[PROGRESS] Starting Bounded Model Checking
Symex completed in: 0.015s (4887 assignments)
Caching time: 0.002s (removed 0 assertions)
Slicing time: 0.019s (removed 4887 assignments)
Generated 0 VCC(s), 0 remaining after simplification (0 assignments)
BMC program time: 0.036s

VERIFICATION SUCCESSFUL
